# Bill Skills — scratch notebook

Ad-hoc analysis against the live Supabase database. This talks to the same
Postgres instance the app uses, from *outside* the app — nothing here runs
as part of the deployed webapp or edge functions.

Copy `notebooks/.env.example` to `notebooks/.env` and fill in real values
before running this. See `CLAUDE.md` for the data model this queries.

In [1]:
import os
import pandas as pd
import psycopg2
from dotenv import load_dotenv

load_dotenv()

conn = psycopg2.connect(os.environ["DATABASE_URL"])

def q(sql, params=None):
    """Run a read query, return a DataFrame. Connects with the DB password,
    which — like the edge functions' service_role key — bypasses RLS, so
    every household is visible here. Scope WHERE clauses deliberately."""
    return pd.read_sql(sql, conn, params=params)

ModuleNotFoundError: No module named 'pandas'

## Investigating a reported issue

Same pattern as the manual `supabase db query --linked` calls used to
diagnose the merchant-mismatch duplicate and the Gmail import failures —
just interactive instead of one-shot. Start from what the user reported
(a merchant, an amount, a date range, a receipt id) and widen from there.

In [2]:
# Example: receipts around a date, across all sources, to spot a duplicate
# or a missing import. Swap the household_id for the one you're debugging.
q("""
    select id, merchant, date, total, source, card_bank, card_last4, created_at
    from public.receipts
    where household_id = %(household_id)s
      and date between %(start)s and %(end)s
    order by date, created_at
""", params={"household_id": "REPLACE_ME", "start": "2026-08-18", "end": "2026-08-22"})

NameError: name 'q' is not defined

In [ ]:
# Example: recent Gmail scan rejections, to see what's being skipped and why
q("""
    select subject, sender, reason, retry_count, scanned_at
    from public.gmail_scan_log
    order by scanned_at desc
    limit 30
""")

## Prototyping a feature before touching the edge functions

For anything that changes `parse-receipt`'s behavior (prompt wording, a new
extracted field, a different model), it's faster to call it directly with a
handful of real `raw_text` samples than to redeploy and re-forward emails
each time. Pull real samples from `raw_ocr_json->>'raw_text'` on existing
receipts so you're testing against real formatting, not a hand-written
example.

In [ ]:
import requests

SUPABASE_URL = os.environ["SUPABASE_URL"]
SERVICE_ROLE_KEY = os.environ["SUPABASE_SERVICE_ROLE_KEY"]

def call_parse_receipt(raw_text: str, reference_date: str | None = None):
    resp = requests.post(
        f"{SUPABASE_URL}/functions/v1/parse-receipt",
        headers={
            "Authorization": f"Bearer {SERVICE_ROLE_KEY}",
            "apikey": SERVICE_ROLE_KEY,
            "Content-Type": "application/json",
        },
        json={"raw_text": raw_text, "reference_date": reference_date},
    )
    resp.raise_for_status()
    return resp.json()

# Grab a real sample to iterate against:
sample = q("""
    select raw_ocr_json->>'raw_text' as raw_text
    from public.receipts
    where source = 'sms' and raw_ocr_json ? 'raw_text'
    limit 1
""").iloc[0]["raw_text"]

call_parse_receipt(sample)

## Sanity-checking a fix against a batch of records

Before writing a migration or a code fix that touches many rows (like the
duplicate-guard change), simulate the new logic here first as a plain
Python/pandas filter over a DataFrame pulled from `q(...)`, and eyeball how
many rows it would flag/change before it ever runs for real.